# satellite_topology_viewer 使用说明

这个模块的核心不是边延迟，而是 **2D 卫星拓扑可视化框架**。

设计原则：

- `module/` 内代码保持参数透明，不写死 G60、XML 路径、delay store 路径。
- 父类 `SatelliteTopology2DViewer` 只关心：有哪些节点、有哪些边、每条边可选的显示值、时间轴、点选、缩放、拖动、region group 显示。
- viewer 只做 GUI 显示，不在内部做 rev-group、拓扑偏移或其他数据变换；需要变换时，先在外部把节点、边、group 数据准备好。
- 子类负责解释边上的值。例如 `EdgeDelayTopologyViewer` 把边值解释为传播时延，并额外显示距离。
- `examples/` 内放可直接运行的演示，比如 G60 的边延迟 GUI。

## 1. 路径准备

In [ ]:
import sys
from pathlib import Path

GENERIC_ROOT = Path(r"E:\paper11\generic")
if str(GENERIC_ROOT) not in sys.path:
    sys.path.insert(0, str(GENERIC_ROOT))

VIEWER_CONFIG = GENERIC_ROOT / "src" / "satellite_topology_viewer" / "examples" / "configs" / "g60_edge_delay_viewer.yaml"
VIEWER_CONFIG

## 2. 直接运行 G60 边延迟 GUI

```powershell
C:\ProgramData\miniconda3\envs\paper11\python.exe E:\paper11\generic\src\satellite_topology_viewer\examples\run_g60_edge_delay_viewer.py
```

快速验证 `0..10s`，不弹窗：

```powershell
C:\ProgramData\miniconda3\envs\paper11\python.exe E:\paper11\generic\src\satellite_topology_viewer\examples\run_g60_edge_delay_viewer.py --start 0 --end 10 --check-only
```

## 3. 父类：只看拓扑和边

`SatelliteTopology2DViewer` 是基础 2D 拓扑 viewer。默认只画黑色边，不要求输入边数值。只有显式传入 `edge_values` 时，才会把边值作为颜色映射来显示。

In [ ]:
import sys
from pathlib import Path

GENERIC_ROOT = Path(r"E:\paper11\generic")
if str(GENERIC_ROOT) not in sys.path:
    sys.path.insert(0, str(GENERIC_ROOT))

from src.config.viewer_config import G60_CONFIG
from src.link_delay.module.edge_options import build_full_option_edges
from src.satellite_topology_viewer.module.base_viewer import SatelliteTopology2DViewer

steps = list(range(0, 11))
edge_table = build_full_option_edges(G60_CONFIG, options=(0, 1, 2, 4))

# 这个单元只演示基础父类需要的数据，不直接弹 GUI。
# 如果要显示窗口，看第 6 节的完整案例。
# viewer = SatelliteTopology2DViewer(
#     G60_CONFIG,
#     steps=steps,
#     edge_table=edge_table,
# )

edge_table.num_edges, len(steps)

## 3.1 基础 GUI 案例：edges + region group

这个单元格不使用 delay store，也不输入边数值。它只准备三类基础输入：

1. `steps`：要显示哪些时间点。
2. `edge_table`：要画哪些边。
3. `group_data`：每个时间点每个 group 对应哪些卫星节点。

因为没有传入 `edge_values`，边默认全是黑色；GUI 只负责显示拓扑、时间轴和 region group。


In [2]:
import sys
from pathlib import Path

# Jupyter/Notebook 中启用 Qt 事件循环；等价于在单元格里写：%gui qt5
try:
    get_ipython().run_line_magic("gui", "qt5")
except NameError:
    pass

GENERIC_ROOT = Path(r"E:\paper11\generic")
if str(GENERIC_ROOT) not in sys.path:
    sys.path.insert(0, str(GENERIC_ROOT))

from PyQt5 import QtWidgets
from src.config.viewer_config import G60_CONFIG
from src.link_delay.module.edge_options import build_full_option_edges
from src.satellite_topology_viewer.module.base_viewer import SatelliteTopology2DViewer
from src.satellite_topology_viewer.module.region_groups import load_or_build_group_data

GROUP_XML = Path(r"E:\paper11\data\basic_file\G60\satellitesposition\station_visible_satellites_20250106.xml")
GROUP_CACHE_DIR = Path(r"E:\paper11\data\basic_file\G60\satellitesposition\full_option_edge_delay\group_data_cache")
EDGE_OPTIONS = (0, 1, 2, 4)
START = 0
END = 86100
STRIDE = 1

# ====================== 数据输入准备（非 GUI 显示层） ======================
# 1) 时间轴
steps = list(range(START, END + 1, STRIDE))

# 2) 要显示的边。这里用全 option 边；如果只想看某些边，改 EDGE_OPTIONS 即可。
edge_table = build_full_option_edges(G60_CONFIG, options=EDGE_OPTIONS)

# 3) region group 数据，格式是 {step: {"groups": {group_id: [node, ...]}}}
group_data = load_or_build_group_data(
    xml_file=GROUP_XML,
    group_cache_dir=GROUP_CACHE_DIR,
    steps=steps,
    station_groups=G60_CONFIG.station_groups,
    total_sats=G60_CONFIG.total_sats,
    constellation_name=G60_CONFIG.name,
    stride=STRIDE,
    enabled=True,
    force=False,
)

# ====================== 纯 GUI 显示层 ======================
# 这里只做 QApplication、创建 viewer、传入已经准备好的数据、show。
app = QtWidgets.QApplication.instance()
if app is None:
    app = QtWidgets.QApplication([])

viewer = SatelliteTopology2DViewer(
    G60_CONFIG,
    steps=steps,
    edge_table=edge_table,
    window_title=f"G60 basic topology edges + groups {START}..{END}s",
    group_data=group_data,
    show_groups=True,
)
viewer.resize(1200, 760)
viewer.show()

# 保留引用，避免 notebook 单元格结束后窗口对象被回收。
_viewer_list = globals().setdefault("_viewer_list", [])
_viewer_list.append(viewer)

print(
    f"basic viewer shown | steps={len(steps)} | edges={edge_table.num_edges} | "
    f"group_steps={len(group_data)} | edge_values=None"
)
viewer


[sat-topology-viewer] Parsing region group data from E:\paper11\data\basic_file\G60\satellitesposition\station_visible_satellites_20250106.xml for 0..86100
[sat-topology-viewer] Wrote group cache: E:\paper11\data\basic_file\G60\satellitesposition\full_option_edge_delay\group_data_cache\station_visible_satellites_20250106_G60_t0_86100_stride1.json
basic viewer shown | steps=86101 | edges=2412 | group_steps=86101 | edge_values=None


## 4. 子类：边延迟 viewer

`EdgeDelayTopologyViewer` 继承父类，只负责把边值解释为 `delay_ms`，并在边详情里显示传播距离。

In [ ]:
import sys
from pathlib import Path

GENERIC_ROOT = Path(r"E:\paper11\generic")
if str(GENERIC_ROOT) not in sys.path:
    sys.path.insert(0, str(GENERIC_ROOT))

from src.config.viewer_config import G60_CONFIG
from src.link_delay.module.edge_options import build_full_option_edges
from src.satellite_topology_viewer.module.base_viewer import build_fake_edge_value_matrix
from src.satellite_topology_viewer.module.edge_delay_viewer import EdgeDelayTopologyViewer

steps = list(range(0, 11))
edge_table = build_full_option_edges(G60_CONFIG, options=(0, 1, 2, 4))
edge_values = build_fake_edge_value_matrix(edge_table, steps)

# 这个单元只演示子类如何接收 delay 矩阵，不直接弹 GUI。
# 如果要显示窗口，看第 6 节的完整案例。
# viewer = EdgeDelayTopologyViewer(
#     G60_CONFIG,
#     steps=steps,
#     edge_table=edge_table,
#     delay_ms=edge_values,
# )

EdgeDelayTopologyViewer.__name__, edge_values.shape

## 5. 从 edge delay store 目录装配 viewer 数据

In [ ]:
import sys
from pathlib import Path

GENERIC_ROOT = Path(r"E:\paper11\generic")
if str(GENERIC_ROOT) not in sys.path:
    sys.path.insert(0, str(GENERIC_ROOT))

from src.config.viewer_config import G60_CONFIG
from src.satellite_topology_viewer.module.edge_delay_data import load_edge_delay_data_for_viewer

STORE_DIR = Path(r"E:\paper11\data\basic_file\G60\satellitesposition\full_option_edge_delay\G60_full_options_t0_86164_stride1")
EDGE_OPTIONS = (0, 1, 2, 4)

delay_data = load_edge_delay_data_for_viewer(
    store_dir=STORE_DIR,
    config=G60_CONFIG,
    start=0,
    end=10,
    stride=1,
    options=EDGE_OPTIONS,
)

delay_data.steps[:3], delay_data.delay_ms.shape

## 6. 完整案例：边延迟 + region group + GUI 显示

这个单元格是完整 GUI 案例，可以单独运行。

它会做这些事：

1. 启动 Jupyter 的 Qt 事件循环。
2. 读取已经生成好的 edge delay store。
3. 读取或复用 region group cache。
4. 创建 `EdgeDelayTopologyViewer` 并显示窗口。

注意：在 Jupyter 里弹 Qt 窗口，需要启用 Qt GUI 集成。代码里使用 `get_ipython().run_line_magic("gui", "qt5")`，等价于手动运行 `%gui qt5`。

下面的代码分成两段：第一段准备输入数据；第二段才是纯 GUI 显示。显示层只消费已经准备好的 `delay_data` 和 `group_data`，不负责计算、偏移或改写这些数据。


In [ ]:
import sys
from pathlib import Path

# Jupyter/Notebook 中启用 Qt 事件循环；等价于在单元格里写：%gui qt5
try:
    get_ipython().run_line_magic("gui", "qt5")
except NameError:
    pass

GENERIC_ROOT = Path(r"E:\paper11\generic")
if str(GENERIC_ROOT) not in sys.path:
    sys.path.insert(0, str(GENERIC_ROOT))

from PyQt5 import QtWidgets
from src.config.viewer_config import G60_CONFIG
from src.satellite_topology_viewer.module.edge_delay_data import load_edge_delay_data_for_viewer
from src.satellite_topology_viewer.module.edge_delay_viewer import EdgeDelayTopologyViewer
from src.satellite_topology_viewer.module.region_groups import load_or_build_group_data

STORE_DIR = Path(r"E:\paper11\data\basic_file\G60\satellitesposition\full_option_edge_delay\G60_full_options_t0_86164_stride1")
GROUP_XML = Path(r"E:\paper11\data\basic_file\G60\satellitesposition\station_visible_satellites_20250106.xml")
GROUP_CACHE_DIR = Path(r"E:\paper11\data\basic_file\G60\satellitesposition\full_option_edge_delay\group_data_cache")
EDGE_OPTIONS = (0, 1, 2, 4)
START = 0
END = 10000
STRIDE = 1

# ====================== 数据输入准备（非 GUI 显示层） ======================
# 1) 读取边延迟数据：形状为 (time, edge)
delay_data = load_edge_delay_data_for_viewer(
    store_dir=STORE_DIR,
    config=G60_CONFIG,
    start=START,
    end=END,
    stride=STRIDE,
    options=EDGE_OPTIONS,
)

# 2) 读取或生成 region group 数据
group_data = load_or_build_group_data(
    xml_file=GROUP_XML,
    group_cache_dir=GROUP_CACHE_DIR,
    steps=delay_data.steps,
    station_groups=G60_CONFIG.station_groups,
    total_sats=G60_CONFIG.total_sats,
    constellation_name=G60_CONFIG.name,
    stride=STRIDE,
    enabled=True,
    force=False,
)

# ====================== 纯 GUI 显示层 ======================
# 这里只做 QApplication、创建 viewer、传入已经准备好的数据、show。
# Jupyter 里不要 app.exec_()，因为 %gui qt5 已经接管事件循环。
app = QtWidgets.QApplication.instance()
if app is None:
    app = QtWidgets.QApplication([])

viewer = EdgeDelayTopologyViewer(
    G60_CONFIG,
    steps=delay_data.steps,
    edge_table=delay_data.edge_table,
    delay_ms=delay_data.delay_ms,
    delay_min_ms=delay_data.delay_min_ms,
    delay_max_ms=delay_data.delay_max_ms,
    window_title=f"G60 edge delay topology {START}..{END}s",
    group_data=group_data,
    show_groups=True,
)
viewer.resize(1200, 760)
viewer.show()

# 保留引用，避免 notebook 单元格结束后窗口对象被回收。
_viewer_refs = globals().setdefault("_viewer_refs", [])
_viewer_refs.append(viewer)

print(
    f"viewer shown | steps={len(delay_data.steps)} | edges={delay_data.edge_table.num_edges} | "
    f"group_steps={len(group_data)} | store={STORE_DIR}"
)
viewer

In [ ]:
%%sql


## 7. 后续扩展方式

如果要做边介数、容量、丢包率等，不改父类。做法是：

1. 准备一个 `(time, edge)` 的矩阵。
2. 继承 `SatelliteTopology2DViewer`。
3. 重写 `format_edge_value()` 和可选的 `edge_extra_description()`。
4. 在 `examples/` 中写一个对应的运行脚本。

In [ ]:
import sys
from pathlib import Path

GENERIC_ROOT = Path(r"E:\paper11\generic")
if str(GENERIC_ROOT) not in sys.path:
    sys.path.insert(0, str(GENERIC_ROOT))

from src.satellite_topology_viewer.module.base_viewer import SatelliteTopology2DViewer

class EdgeBetweennessTopologyViewer(SatelliteTopology2DViewer):
    def format_edge_value(self, edge_idx: int, row: int, value: float) -> str:
        return f"edge_betweenness={value:.0f} paths"

# 之后把 edge_values 换成边介数矩阵即可。
EdgeBetweennessTopologyViewer.__name__